# Example: Breast Cancer Diagnosis

This demonstration uses the **GradFlow** engine to solve a **Breast Cancer Diagnosis** classification problem.

### 1. Imports and environment setup

In [23]:
import random
from gradflow.utils import read_csv
from gradflow.tensor import Tensor
from gradflow.nn.mlp import MLP

random.seed(1337)

### 2. Dataset Preparation
We use our custom `read_csv` utility to load the clinical data. The labels are `1.0` (Malignant) and `-1.0` (Benign).

In [24]:
data = read_csv('medical_data.csv')
print(f"Loaded {len(data)} clinical samples.")

features = []
labels = []

for row in data:
    features.append([
        float(row['radius_mean']), 
        float(row['texture_mean']), 
        float(row['smoothness_mean'])
    ])
    labels.append(float(row['diagnosis']))

Loaded 10 clinical samples.


### 3. Model Architecture
Our model uses Gaussian initialization to ensure neurons start in an active state.

In [25]:
model = MLP(3, [6, 6, 1])
print(f"Network initialized with {len(model.parameters())} parameter Tensors.")

Network initialized with 26 parameter Tensors.


### 4. Training Engine
We define our Batch Gradient Descent update rule here.

In [26]:
def update_weights(model_parameters, learning_rate):
    for p in model_parameters:
        def update(item):
            if isinstance(item, list):
                for x in item: update(x)
            else: # Value object
                item.data -= learning_rate * item.grad
        update(p.data)

### 5. Training Loop
We train for 500 epochs with a learning rate of 0.01.

In [27]:
epochs = 500
learning_rate = 0.01

for i in range(epochs):
    predictions = [model(x) for x in features]
    # prediction.data[0] is the scalar Value output of the linear unit
    loss = sum((prediction.data[0] - target_label)**2 for target_label, prediction in zip(labels, predictions)) * (1.0 / len(labels))
    
    model.zero_grad()
    loss.backward()
    
    update_weights(model.parameters(), learning_rate)
    
    if i % 100 == 0 or i == epochs - 1:
        print(f"Epoch {i:3d} | Loss: {loss.data:.4f}")

Epoch   0 | Loss: 2.6250
Epoch 100 | Loss: 0.0399
Epoch 200 | Loss: 0.0179
Epoch 300 | Loss: 0.0173
Epoch 400 | Loss: 0.0169
Epoch 499 | Loss: 0.0166


### 6. Clinical Inference
Final model validation.

In [29]:
print("True Label | Pred (Raw) | Diagnosis")
print("------------------------------------")
for x, y_true in zip(features[:6], labels[:6]):
    out = model(x).data[0].data
    diagnosis = "MALIGNANT" if out > 0 else "BENIGN"
    print(f"{y_true:10.1f} | {out:10.4f} | {diagnosis}")

True Label | Pred (Raw) | Diagnosis
------------------------------------
      -1.0 |    -0.9767 | BENIGN
      -1.0 |    -1.0088 | BENIGN
       1.0 |     0.9382 | MALIGNANT
       1.0 |     1.1181 | MALIGNANT
      -1.0 |    -0.9936 | BENIGN
       1.0 |     0.7952 | MALIGNANT
